<a href="https://colab.research.google.com/github/patriciasandagorda/UA_MDM_Labo2_Grupo2/blob/ramaPatricia/06_kaggle_texto_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este script permite descargar datos de una competencia de Kaggle directamente en Google Colab.

Requiere que el usuario genere un token de autenticación de Kaggle (kaggle.json)
y lo suba al entorno de Colab para autenticar la descarga.

Guía para generar el token:
1. Ir a https://www.kaggle.com
2. En la sección 'API', hacer clic en 'Create New API Token'
3. Esto descargará un archivo llamado `kaggle.json` en tu computadora
4. Luego, en Colab, deberás subir este archivo como se muestra en las instrucciones siguientes

Guía para Generar token y descargar: https://www.kaggle.com/discussions/general/74235.


In [ ]:
# Instala librerías
!pip install optuna
!pip install -q kaggle

from google.colab import files

In [ ]:
# Cargar el token de Kaggle (json)
files.upload()

In [ ]:
 # Crea el directorio de configuración de Kaggle si no existe
 ! mkdir ~/.kaggle

 # Copia el archivo `kaggle.json` al directorio de configuración
 ! cp kaggle.json ~/.kaggle/

 # Establece los permisos de acceso al file
 ! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Lista datasets públicos disponibles (a modo de validación)
! kaggle datasets list

In [ ]:
# Descarga los datos de la competencia
! kaggle competitions download -c 'petfinder-adoption-prediction'

In [ ]:
# Crea la carpeta input
! mkdir input
# Crea un subcarpeta de la competencia
! mkdir ./input/petfinder-adoption-prediction
# Mueve el archivo ZIP descargado a la carpeta input
! mv petfinder-adoption-prediction.zip ./input
# Descomprime el archivo ZIP
! unzip ./input/petfinder-adoption-prediction.zip -d ./input/petfinder-adoption-prediction/

In [ ]:
#06_text_classification

Fuentes: https://medium.com/nlplanet/fine-tuning-distilbert-on-senator-tweets-a6f2425ca50e

#### **Instalar Modulos**

conda install datasets=="2.20.0"

conda install transformers=="4.40.1"

conda install numpy=="1.26.4" # La última versión no funciona bien


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
#from UA_MDM_LDI_II.tutoriales.utils import plot_confusion_matrix

# Modeling
import torch
from torch.optim import AdamW # Moved AdamW import from transformers to torch.optim
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding, AutoModelForSequenceClassification, get_scheduler

# Progress bar
from tqdm.auto import tqdm

# Verificamos que CUDA está funcional
torch.cuda.is_available()

**Bajamos el modelo**

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')  #19.11   5 categorias disntintas

**Armado de los Datasets**

In [ ]:
# Paths
BASE_DIR = '/content/'
PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parametros y variables
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 64

MODEL_NAME = '06 Bert'

MODEL_VERSION = '1.0'

In [ ]:
# Cargar los datos
df = pd.read_csv(PATH_TO_TRAIN)
df = df[df['Description'].notnull()] #que pasa con los datos que tienen la descripcion nula?
df['labels'] = df["AdoptionSpeed"]


In [ ]:
train_df.head()

In [ ]:

# Dividir los datos usando sklearn
#train_df, test_df = train_test_split(train_df, test_size=TEST_SIZE, random_state=SEED, stratify=train_df.AdoptionSpeed)

study_lgb = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name="04 - LGB Multiclass CV",
                           load_if_exists = True)


lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

train_df = df[~df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)
test_df = df[df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)

# Convertir a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Combinar en un DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Hacer una lista de columnas para remover antes de la tokenización
cols_to_remove = [col for col in dataset["train"].column_names if col != 'labels']
print(cols_to_remove)

In [ ]:
# Obtener el objeto ClassLabel del conjunto de datos de entrenamiento
class_label = dataset["train"].features["labels"] # donde se usa???

# Obtener las clases originales a partir del objeto ClassLabel
classes = class_label.names
classes

In [ ]:
# Tokenize and encode the dataset
def tokenize(batch):
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokenized_batch = tokenizer(batch["Description"], padding=True, truncation=True, max_length=512)
    return tokenized_batch

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=4)

# Set dataset format for PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check the output
print(dataset_enc["train"].column_names)
print(dataset_enc["test"].column_names)


In [ ]:
# Instantiate a data collator with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create data loaders for to reshape data for PyTorch model
train_dataloader = DataLoader(
    dataset_enc["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc["test"], batch_size=BATCH_SIZE, collate_fn=data_collator
)

In [ ]:
# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                           num_labels=num_labels) #19:11

In [ ]:
# Model parameters ---- codigo sin optuna
########OJO SIN OPTUNA



# learning_rate = 5e-5   #19.17
# num_epochs = 15


# # Create the optimizer
# optimizer = AdamW(model.parameters(), lr=learning_rate)    #aca hace el backpropágation 19.15
# #ver explicacion!!!!! para reducir el error-- el optimizador decide si hace descenso de gradiente o si tiene una inercia, etc..
# #no es lo mismo que optuna(ver otro cuaderno)

# # Further define learning rate scheduler
# num_training_batches = len(train_dataloader)
# num_training_steps = num_epochs * num_training_batches
# lr_scheduler = get_scheduler(
#     "linear",                   # linear decay
#     optimizer=optimizer,
#     num_warmup_steps=0,
#     num_training_steps=num_training_steps,
# )



**Miramos el Modelo**

In [ ]:

# Set the device automatically (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Move model to device
model.to(device)


 #768 features--- 19.13
#score que le da a todas la velocidad

# 19.14 (ventana de contexto) tamaño 512--- transoremen da tamaño maximo de texto que l epuede poner--- !!!
#se tokeniza la variable descripcion, no el dataset


In [ ]:
def train_val(model, dataloaders, datasets, device, num_epochs=4, lr=0.001, trial=None):

    since = time.time()

    # Create the optimizer
    optimizer = AdamW(model.parameters(), lr=lr)

    # Further define learning rate scheduler
    num_training_batches = len(train_dataloader)
    num_training_steps = num_epochs * num_training_batches
    lr_scheduler = get_scheduler(
        "linear",                   # linear decay
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )


    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999

    train_losses = []
    val_losses = []

    try:
        previous_best = study.best_value
    except:
        previous_best = -999


    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        kappa_labels_true = []
        kappa_labels_predicted = []
        output_scores = []

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for batch in tqdm(dataloaders[phase]):
                batch = batch.to(device)
                #inputs = inputs.to(device)
                labels = batch.labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(**batch)
                    loss = outputs.loss

                    preds = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    preds_labels = torch.argmax(preds, dim=-1)


                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                    elif phase == 'val':
                        kappa_labels_true.extend(labels.cpu().numpy().tolist())
                        kappa_labels_predicted.extend(preds_labels.cpu().numpy().tolist())
                        outputs_np = preds.cpu().numpy()
                        output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                # Statistics
                running_loss += loss.item() * labels.size(0)
                running_corrects += torch.sum(preds_labels == labels.data)

                #END OF BATCH

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc = running_corrects.double() / len(datasets[phase])

            if phase == 'train':
                train_losses.append(epoch_loss)
                kappa_score = np.nan
            else:
                val_losses.append(epoch_loss)
                kappa_score = cohen_kappa_score(kappa_labels_true,
                                  kappa_labels_predicted,
                                  weights = 'quadratic')



            print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {kappa_score:.3f}')

            # If this is the best Epoch so far -> Deep copy the model
            if phase == 'val' and kappa_score > best_kappa:
                best_acc = epoch_acc
                best_kappa = kappa_score
                best_model_wts = copy.deepcopy(model.state_dict())


                #Best Epoch within a trial and better than previous trials
                if trial is not None and best_kappa > previous_best:

                    #Save test dataset with predictions
                    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                    predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                'pred':output_scores}).merge(test_df, on='PetID')
                    dump(predicted_df, predicted_filename)

                    #Generate and save CM
                    cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                    plot_confusion_matrix(kappa_labels_true,kappa_labels_predicted).write_image(cm_filename)

            #END OF PHASE

        #END OF EPOCH

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:.2f}%'.format(best_acc * 100))

    # Load best model weights
    model.load_state_dict(best_model_wts)

    # Save in optuna trial the best test dataset, cm and model weights
    if trial is not None and best_kappa > previous_best:
        upload_artifact(trial, predicted_filename, artifact_store)

        upload_artifact(trial, cm_filename, artifact_store)

        file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
        model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
        torch.save(model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
        upload_artifact(trial, model_path, artifact_store)

    return model,best_kappa



In [ ]:

# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")


In [ ]:

best_model,_ = train_val(model,
                       dataloaders={'train': train_dataloader,
                                    'val': eval_dataloader},
                       datasets=dataset_enc,
                       device=device,
                       lr = 5e-5,
                       num_epochs=15)


In [ ]:
# Guardo el modelo
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
print(f'Modelo guardado en {model_path}')

In [ ]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def optuna_train(trial):

    epochs = trial.suggest_int('epochs', 1, 2)

    lr = trial.suggest_float('lr', 0.00001, 0.0001, log=True)

    _,best_score = train_val(model,
                       dataloaders={'train': train_dataloader,
                                    'val': eval_dataloader},
                       datasets=dataset_enc,
                       device=device,
                       num_epochs=epochs,
                       lr=lr,
                       trial=trial)


    return(best_score)

In [ ]:
study = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)
study.optimize(optuna_train, n_trials=5)

**Entrenamos**

In [ ]:
###### apartir de aca es el codigo sin optuna. Reutilizar kappa

In [ ]:
progress_bar = tqdm(range(num_training_steps))

# Train the model with PyTorch training loop
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)    #en maquina 20 min 19.18

**Obtenemos la kappa base**

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Inicializa listas para almacenar todas las predicciones y etiquetas
all_predictions = []
all_labels = []

# Iteratively evaluate the model and collect predictions and labels
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    # Mover predicciones y etiquetas a CPU y convertir a numpy
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

# Convertir listas a arrays de numpy
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Calcular Quadratic Weighted Kappa
qwk = cohen_kappa_score(all_labels, all_predictions, weights='quadratic')

print(f"Quadratic Weighted Kappa: {qwk}")


**Predecimos un ejemplo de descripción**

In [ ]:
# Un ejemplo
desc = test_df.iloc[4]['Description']
print(desc)

# Tokenize inputs
inputs = tokenizer(desc, padding=True, truncation=True, return_tensors="pt").to(device) # Move the tensor to the GPU

# Inference model and get logits
outputs = model(**inputs)

# Convert logits to class probabilities
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
probabilities = predictions.detach().cpu().numpy()
predicted_class = np.argmax(probabilities, axis=1)

# Establecer opciones de impresión para evitar la notación científica
np.set_printoptions(suppress=True, formatter={'float_kind': '{:.8f}'.format})
print(probabilities[0])
print(predicted_class)

#aca da ulas probalidadde de una descripvion

In [ ]:
# Define el directorio donde quieres guardar el modelo
output_dir = './my_distilbert_model'

# Guarda el modelo y el tokenizer en el directorio especificado
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Modelo y tokenizer guardados en: {output_dir}")

# Para descargar a tu computadora local, busca la carpeta 'my_distilbert_model'
# en el explorador de archivos de Colab (panel izquierdo) y descárgala.

PARA CARGAR MODELO ENTRENADO

In [ ]:
#para cargar el modelo entrenado!
from transformers import AutoModelForSequenceClassification, DistilBertTokenizerFast

# Define el directorio donde guardaste el modelo
output_dir = './my_distilbert_model' # O la ruta completa a la carpeta

# Carga el tokenizer
loaded_tokenizer = DistilBertTokenizerFast.from_pretrained(output_dir)

# Carga el modelo
loaded_model = AutoModelForSequenceClassification.from_pretrained(output_dir)

print("Modelo y tokenizer cargados exitosamente!")

# Ahora puedes usar loaded_tokenizer y loaded_model para hacer inferencias.